PRZYGOTOWANIE:  instalacja potrzebnych bibliotek oraz import plików


In [ ]:
!pip3 install morfeusz2 pdfplumber nltk stop_words wordcloud

In [ ]:
import os
import pandas as pd
import pickle
import re
import pdfplumber
import numpy as np
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import morfeusz2
from stop_words import get_stop_words
from collections import defaultdict
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import seaborn as sns

Nad projektem pracowałem częściowo w swoim środowisku lokalnym, a częściowo w google colab (bo mój Macbook za długo pracował z modelem BERT), stąd na różnych etapach potrzebowałem wczytać różne pliki, niektóre już były przetworzone za pomocą pipeline poniżej przedstawionego, więc niepotrzebnie było robić to po parę razy. Najbardziej pierwotnym plikiem do pracy był "annotated_texts_8_term.csv", który był plikiem outputowym z notebooka "stenogramy_obrobka_danych.ipynb".

In [ ]:
entries = pd.read_csv("annotated_texts_8_term.csv") #output z stenogramy_obrobka_danych.ipynb
#entries = pd.read_csv("przetworzone_stenogramy.csv")
#df_lemma = pd.read_csv("slownik_lematow.csv")

nawl = pd.read_csv("nawl-analysis.csv") #Nencki Affetictive Word List
#entries = pd.read_csv("wypowiedzi_labeled.csv") #wczytywane do późniejszej pracy, gdy już było obrobione tym notebookiem

Sprawdzenie czy działa i jak wyświetla.

In [ ]:

entries

In [ ]:
entries.columns

Wypełniamy puste wypowiedzi i tokenizujemy dla języka polskiego.

In [ ]:
entries['text'] = entries['text'].fillna("")
entries['text_tokenized'] = entries['text'].apply(word_tokenize, language='polish')


In [ ]:
entries['text_tokenized']

Zrobienie unikalnych tokenów.

In [ ]:
all_tokens = [token for sublist in entries['text_tokenized'] for token in sublist]
unique_tokens = list(set(all_tokens))

Lematyzacja za pomocą Morfeusz (morfeusz.sgjp.pl), inspirowana kodem z notebooków dot. alimentów.  

In [ ]:
morf = morfeusz2.Morfeusz()
lemma_dict = {}
#dodaje pasek postępu żeby widzieć ze działa i się robi
for token in tqdm(unique_tokens):
    try:
        #analiza morfeuszem
        analysis = list(morf.analyse(token))
        if analysis:
            # pierwszy wynik analizy jest najbardziej prawdopodobny
            outer_tuple = analysis[0]
            # to jest zawartość językowa
            linguistic_data = outer_tuple[2]
            # tu jest surowy lemat
            lemma_raw = linguistic_data[1]
            # wyczyszczenie żeby nie miał tagów
            lemma_clean = lemma_raw.split(':')[0]
            lemma_dict[token] = lemma_clean
        else:
            lemma_dict[token] = token
    except Exception:
        #pozostaje oryginał jeśli coś nie zadziała
        lemma_dict[token] = token



# sprawdzam jak się zlematyzowały
print("Słowo->Lemat:")
for k, v in list(lemma_dict.items())[:10]:
    print(f"{k} -> {v}")

Zapis do pliku "slownik_lematow.csv" aby wykorzystać później i nie musieć uruchamiać całości pipeline od początku.

In [ ]:
#przepisanie słownika do DataFrame
df_lemma = pd.DataFrame(list(lemma_dict.items()), columns=['original', 'lemma'])

#zapisanie do pliku aby uruchomić później
df_lemma.to_csv('slownik_lematow.csv', index=False, encoding='utf-8')

Zamieniamy wyrazy w stokenizowanym tekście na lematy zgodne ze słownikiem lematów dla niniejszej analizy.

In [ ]:
# zamieniam tokeny na lematy
def replace_with_lemmas(tokens_list):
    return [lemma_dict.get(token, token) for token in tokens_list]

#kolumna z listą lematów
entries['text_lemmatized_list'] = entries['text_tokenized'].apply(replace_with_lemmas)

#kolumna ze stringiem z listy lematów - czasami tak bardziej przydatne do analizy
entries['text_lemmatized_str'] = entries['text_lemmatized_list'].str.join(' ')

#sanity check kilku przykładowych
print("Oryginał:", entries['text_tokenized'].iloc[3][:10])
print("Lematy:  ", entries['text_lemmatized_list'].iloc[3][:10])
print("Tekst zlematyzowany:   ", entries['text_lemmatized_str'].iloc[3][:100])
print("Tekst oryginalny:    ", entries['text'].iloc[3][:100])

Znowu zapisujemy do pliku, tym razem 'przetworzone_stenogramy.csv'.

In [ ]:
#tutaj eksport do pliku tego co zostało przetworzone aby wykorzystać dalej
columns_to_save = ['doc_id', 'name', 'role', 'district','club', 'text_lemmatized_str']
entries[columns_to_save].to_csv('przetworzone_stenogramy.csv', index=False, encoding='utf-8')


Lematyzowanie stop words.

In [ ]:
#lista polskich stop words
stop_words_list = get_stop_words('pl')

#kilka dodanych stop words do listy (po analizie dopiero rozszerzono o dalsze)
custom_stops = ['głos', 'poseł', 'pani', 'pan', 'wysoka', 'izba', 'marszałek', 'proszę', 'dziękuję',
                'projekt', 'ustawa', 'posiedzenie', 'unia', 'europejska', 'porządek', 'dzienny', 'sprawozdanie',
                'komisja', 'głosować', 'ręka', 'nacisnąć', 'przycisk', 'przeciw', 'wstrzymywać', 'klub', 'zechcieć']
stop_words_list.extend(custom_stops)

#lematyzacja stop words
def lemmatize_stopword(word):
    # usuwanie wszystkiego poza literami
    clean_word = re.sub(r'[^a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]', '', word).lower()
    # lematyzacja
    return lemma_dict.get(clean_word, clean_word)

# ostateczna lista zlematyzowanych stop words
final_stop_words = set([lemmatize_stopword(w) for w in stop_words_list])

Czyszczenie ze stop words i generowanie ngramów.

In [ ]:
def generate_ngrams_and_clean(tokens_list):
    if not isinstance(tokens_list, list):
        return []
    # usuwanie stop words
    clean_tokens = [t for t in tokens_list if t not in final_stop_words]
    # unigramy
    unigrams = clean_tokens
    # bigramy
    bigrams = ['_'.join(pair) for pair in zip(clean_tokens, clean_tokens[1:])]
    # trigramy
    trigrams = ['_'.join(trio) for trio in zip(clean_tokens, clean_tokens[1:], clean_tokens[2:])]
    # zwracanie wszystkiego (zdecydowano w którejś wersji, że lepiej bez unigramów bo trochę zaśmiecają)
    return bigrams + trigrams

#dopisanie ngramów do tabeli entries
entries['ngrams'] = entries['text_lemmatized_list'].apply(generate_ngrams_and_clean)

# przykładowe ngramy
print(entries['ngrams'].iloc[9][-15:])

Zliczanie najczęściej występujących ngramów oraz zrobienie z nich Word Cloud.

In [ ]:
#do zignorowania - często wskakiwało, błąd który pojawił się na samym końcu przy ostatnim uruchamianiu skryptu
ignore_set = {'-', '–', '—', '−', '.', ',', '_'}
junk_chars = '-–—−_., '

#wszystkie ngramy
all_words = [word.strip(junk_chars) for sublist in entries['ngrams'] for word in sublist if word not in ignore_set]

#zliczenia ngramów
word_counts = Counter(all_words)

#50 najpopularniejszych ngramów
common_words = word_counts.most_common(30)
words, freqs = zip(*common_words)
#wyświetlanie
plt.figure(figsize=(12, 8))
sns.barplot(x=list(freqs), y=list(words), palette='viridis')
plt.title("30 najczęstszych ngramów")
plt.xlabel("Liczba wystąpień")
plt.show()

#wordcloud
wordcloud = WordCloud(width=1600, height=800,background_color='white',colormap='viridis',max_words=200).generate_from_frequencies(word_counts)
plt.figure(figsize=(15, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Stenogramy Sejmowe", fontsize=20)
plt.show()

Konkluzja tutaj jest raczej jasna - bez nawet wstępnego przefiltrowania po słowach z emocjami w dyskursie parlamentarnym najczęściej występują frazy związane z technikaliami procesu legislacyjnego.

In [ ]:
print(common_words)

Histogram rozkładu długości wypowiedzi w Sejmie.

In [ ]:
#rozkład długości wypowiedzi sejmowych
doc_lengths = entries['text_lemmatized_str'].apply(len)

#binowanie co 100
max_length = doc_lengths.max()
bins_range = range(0, max_length + 500, 100)

#histogram
plt.figure(figsize=(12, 6))
plt.hist(doc_lengths, bins=bins_range, color='skyblue', edgecolor='black')
plt.title("Rozkład długości wypowiedzi")
plt.xlabel("Liczba słów w wypowiedzi")
plt.ylabel("Liczba wypowiedzi")
plt.xlim(0, 15000)
plt.grid(axis='y', alpha=0.5)
plt.show()

Obróbka i przystosowanie pliku z Nencki Affective Word List do potrzeb.

In [ ]:
#obróbka nawl
nawl.columns = nawl.columns.str.strip()
nawl['word'] = nawl['word'].str.lower().str.strip()

# mapowanie kolumn - będą potrzebne dla każdej wypowiedzi
columns_map = {
    # Happiness
    'mean Happiness': 'H_Mean',
    'distance to H':  'H_Dist',
    # Anger
    'mean Anger':     'A_Mean',
    'distance to A':  'A_Dist',
    # Sadness
    'mean Sadness':   'S_Mean',
    'distance to S':  'S_Dist',
    # Fear
    'mean Fear':      'F_Mean',
    'distance to F':  'F_Dist',
    # Disgust
    'mean Disgust':   'D_Mean',
    'distance to D':  'D_Dist',
    # Neutral
    'distance to N':  'N_Dist'
}

#
cols_to_use = list(columns_map.keys())

nawl_advanced = nawl.set_index('word')[cols_to_use] \
                    .rename(columns=columns_map) \
                    .to_dict(orient='index')


Funkcja do Dictionary/Corpus Based Sentiment Analysis - zlicza sumę metryk w poszczególnych kolumnach dla listy tokenów z wypowiedzi, a następnie normalizuje przez ilość znalezionych słów.

In [ ]:
def get_advanced_emotions(tokens_list):
    # konfiguracja metryk - zgodnie z columns_map
    target_metrics = [
        'H_Mean', 'H_Dist',
        'A_Mean', 'A_Dist',
        'S_Mean', 'S_Dist',
        'F_Mean', 'F_Dist',
        'D_Mean', 'D_Dist',
        'N_Dist'
    ]

    # liczymy sumy dla poszczególnych metryk w wypowiedzi
    sums = {metric: 0.0 for metric in target_metrics}

    match_count = 0  # potrzebne do normalizacji

    if not isinstance(tokens_list, list) or len(tokens_list) == 0:
        return {metric: 0.0 for metric in target_metrics}

    #iterujemy po każdym tokenie
    for token in tokens_list:
        #jeśli token jest w nawl
        if token in nawl_advanced:
            values = nawl_advanced[token]
            match_count += 1
            for metric in target_metrics:
                val = values.get(metric, 0.0)
                if pd.notnull(val):
                    sums[metric] += val

    # jeśli nie mamy słów z nawl w wypowiedzi to metryki są równe 0
    if match_count == 0:
        return {metric: 0.0 for metric in target_metrics}

    # uśredniamy metryki
    averages = {}
    for metric, total_value in sums.items():
        averages[metric] = total_value / match_count

    # dodajemy metadaną
    averages['nawl_word_count'] = match_count

    return averages

Corpus/dictionary bases sentiment analysis - używamy słów emocjonalnych z NAWL do klasyfikacji sentymentu wypowiedzi.

In [ ]:
#tekst zamieniany na listę słów (bo w niektórych wczytywanych plikach jedno jest a drugiego nie ma)
entries['text_lemmatized_list'] = entries['text_lemmatized_str'].str.split()
print("Początek analizy")
#używamy wyżej wymienionej funkcji do analizy
advanced_emotions_series = entries['text_lemmatized_list'].apply(get_advanced_emotions)
#robimy z tego DataFrame
advanced_emotions_df = pd.DataFrame(advanced_emotions_series.tolist())
#ponownie w jednych plikach było a w drugich nie, więc trzeba było wyczyscić i zrobić nowe
existing_cols = [c for c in advanced_emotions_df.columns if c in entries.columns]
entries = entries.drop(columns=existing_cols, errors='ignore')
#połączenie z właśnie zastosowaną analizą
entries = pd.concat([entries, advanced_emotions_df], axis=1)

cols_to_show = ['text_lemmatized_str', 'H_Mean', 'H_Dist', 'A_Mean', 'A_Dist']
print(entries[cols_to_show].head())

Średnie natężenie złości (Anger), top 10 wypowiedzi z największym poziomem wstrętu (Disgust)

In [ ]:

if 'club' in entries.columns:
    print("Średnie natężenie Złości (Anger Mean) w klubach parlamentarnych")
    ranking_zlosci = entries.groupby('club')['A_Mean'].mean().sort_values(ascending=False) #funkcja agregująca group by po klubie
    print(ranking_zlosci)



In [ ]:
print("-" * 30)
print("Top 15 angry wypowiedzi")

top_disgust = entries.sort_values(by='A_Mean', ascending=False).head(15) #sortowanie po średniej Anger w wypowiedzi

for idx, row in top_disgust.iterrows():
    print(f"\nPoseł: {row['name']} (Klub: {row.get('club', 'brak')})")

    print(f"\nParametry(Natężenie): {row['A_Mean']:.2f} | (Dystans): {row['A_Dist']:.2f}")

    tekst = row.get('text', row.get('text', 'Brak tekstu'))
    print(f"\nwypowiedź: {str(tekst)[:]}...")
    print("-" * 30)

In [ ]:
print("-" * 30)
print("Top 15 disgusting wypowiedzi")

top_disgust = entries.sort_values(by='D_Mean', ascending=False).head(15) #sortowanie po średniej Disgust w wypowiedzi

for idx, row in top_disgust.iterrows():
    print(f"\nPoseł: {row['name']} (Klub: {row.get('club', 'brak')})")

    print(f"\nParametry(Natężenie): {row['D_Mean']:.2f} | (Dystans): {row['D_Dist']:.2f}")

    tekst = row.get('text', row.get('text', 'Brak tekstu'))
    print(f"\nwypowiedź: {str(tekst)[:]}...")
    print("-" * 30)

In [ ]:
print("-" * 30)
print("Top 15 smutnych wypowiedzi")

top_disgust = entries.sort_values(by='S_Mean', ascending=False).head(15) #sortowanie po średniej Sadness w wypowiedzi

for idx, row in top_disgust.iterrows():
    print(f"\nPoseł: {row['name']} (Klub: {row.get('club', 'brak')})")

    print(f"\nParametry(Natężenie): {row['S_Mean']:.2f} | (Dystans): {row['S_Dist']:.2f}")

    tekst = row.get('text', row.get('text', 'Brak tekstu'))
    print(f"\nwypowiedź: {str(tekst)[:40]}...")
    print("-" * 30)

Powyżej przykładowe wyniki najsilniejszych emocji z poszczególnej kategorii.

In [ ]:

mean_cols = ['H_Mean', 'A_Mean', 'S_Mean', 'F_Mean', 'D_Mean']

#funkcja agregująca grupowanie po klubie
club_means = entries.groupby('club')[mean_cols].mean()

#normalizowanie udziału emocji do 100%
club_means_100 = club_means.div(club_means.sum(axis=1), axis=0) * 100

labels = {
    'H_Mean': 'Radość',
    'A_Mean': 'Złość',
    'S_Mean': 'Smutek',
    'F_Mean': 'Strach',
    'D_Mean': 'Wstręt'
}
club_means_100 = club_means_100.rename(columns=labels)

#wybór kolorów
colors_map = {
    'Radość': '#F4D03F',
    'Złość': '#E74C3C',
    'Smutek': '#3498DB',
    'Strach': '#7F8C8D',
    'Wstręt': '#27AE60'
}
plot_colors = [colors_map.get(col, '#333') for col in club_means_100.columns]

#wykres
plt.figure(figsize=(14, 8))
ax = club_means_100.plot(
    kind='bar',
    stacked=True,
    color=plot_colors,
    figsize=(14, 8),
    width=0.8,
    edgecolor='white'
)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.title('Profil emocjonalny wypowiedzi w podziale na Kluby Parlamentarne', fontsize=16)
plt.ylabel('Udział procentowy (%)', fontsize=12)
plt.xlabel('Klub Parlamentarny', fontsize=12)
plt.legend(title='Emocje', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

Ciekawiej natomiast to wygląda w modelu, w którym grupujemy emocje na 3 grupy: pozytywne (H), aktywne negatywne (A, D), pasywne negatywne (S, F) - widać wtedy wyraźniej różnice pomiędzy poszczególnymi klubami parlamentarnymi.

In [ ]:
entries.to_csv("wypowiedzi_labeled.csv")

In [ ]:

base_cols = ['H_Mean', 'A_Mean', 'D_Mean', 'S_Mean', 'F_Mean']
#grupowanie po klubie
club_means = entries.groupby('club')[base_cols].mean()

#nowe emocje - pogrupowane
grouped_emotions = pd.DataFrame(index=club_means.index)

#pozotywne
grouped_emotions['Pozytywne (Radość)'] = club_means['H_Mean']

#aktywne negatywne
grouped_emotions['Wrogość (Złość + Wstręt)'] = club_means['A_Mean'] + club_means['D_Mean']

#pasywne negatywne
grouped_emotions['Niepokój (Smutek + Strach)'] = club_means['S_Mean'] + club_means['F_Mean']

#normalizacja do 100
grouped_emotions_100 = grouped_emotions.div(grouped_emotions.sum(axis=1), axis=0) * 100

# sortowanie od najbardziej pozytywnych partii
grouped_emotions_100 = grouped_emotions_100.sort_values(by='Pozytywne (Radość)', ascending=True)

# kolory
group_colors = {
    'Pozytywne (Radość)': '#F4D03F',
    'Wrogość (Złość + Wstręt)': '#C0392B',
    'Niepokój (Smutek + Strach)': '#5D6D7E'
}
plot_colors = [group_colors[col] for col in grouped_emotions_100.columns]

plt.figure(figsize=(14, 8))
ax = grouped_emotions_100.plot(
    kind='bar',
    stacked=True,
    color=plot_colors,
    figsize=(14, 8),
    width=0.85,
    edgecolor='white',
    linewidth=1
)

plt.title('Profil emocjonalny Klubów: Pozytywne vs Wrogość vs Niepokój', fontsize=16)
plt.ylabel('Udział w profilu emocjonalnym (%)', fontsize=12)
plt.xlabel('Klub Parlamentarny', fontsize=12)
plt.legend(title='Grupa emocji', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.axhline(y=50, color='black', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:

print("\nSzczegółowe dane (procentowy udział grup):")
print(grouped_emotions_100.round(2))

**CZĘŚĆ MACHINE LEARNING**

Instalowanie modułu transformers. Będe korzystać z modelu *bert-polish-sentiment-politics*. Dwie poniższe komórki z instalacją są przekopiowane z notebooka tutorialowego do instalacji.

In [ ]:
!pip install -U transformers

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="eevvgg/bert-polish-sentiment-politics")
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("eevvgg/bert-polish-sentiment-politics")
model = AutoModelForSequenceClassification.from_pretrained("eevvgg/bert-polish-sentiment-politics")

Faktyczne analiza sentymentu.

In [ ]:
sample_size = 8192

#losowanie próbki
#ze względu na niezbyt sprawny laptop do ML oraz ograniczoną ilość zasobów obliczeniowych w Google Colab, użyłem ok. 1/8 wszystkich wypowiedzi, co zajęło ok. 35 min.
#początkowo sprawdziłem czas obliczeń dla 256 przykładów i na bazie tego uznałem że 8192 to limit przy którym Colab nie odłączy środowiska
entries_sample = entries.sample(n=real_n, random_state=42).copy()

texts_to_process = entries_sample['text_lemmatized_str'].fillna("").astype(str).tolist()

#wielkość próbki
batch_size = 16
results = []

#wyświetlanie z paskiem, żeby widzieć postęp, skracam do 512 bo tyle tokenów obsługuje bert maksymalnie, więc tu tracona jest informacja
for i in tqdm(range(0, len(texts_to_process), batch_size)):
    batch = texts_to_process[i : i + batch_size]
    try:
        batch_results = pipe(batch, truncation=True, max_length=512)
        results.extend(batch_results)
    except Exception as e:
        print(f"Błąd w {i}: {e}")
        results.extend([{'label': 'neutral', 'score': 0.0}] * len(batch))

#normalizacja wyników
def parse_bert_result(result):
    label = result['label'].lower()
    score = result['score']

    mapping = {
        'label_0': 'negative',
        'label_1': 'neutral',
        'label_2': 'positive',
        'negative': 'negative',
        'neutral': 'neutral',
        'positive': 'positive'
    }
    return mapping.get(label, 'unknown'), score

# parsowanie wyników
parsed_data = [parse_bert_result(r) for r in results]
# dataframe z wynikami
bert_df = pd.DataFrame(parsed_data, columns=['BERT_Label', 'BERT_Score'])

# połączenie indeksów losowej próbki z właściwymi indeksami z entries
entries_sample = entries_sample.reset_index(drop=True)
entries_sample = pd.concat([entries_sample, bert_df], axis=1)

print(entries_sample[['name', 'BERT_Label', 'BERT_Score']].head())

In [ ]:
#prosta funkcja etykietująca: jeśli suma negatywnych emocji jest większa niż suma pozytywnych, to jest to wypowiedź negatywna, a w p.p. pozytywna
def get_sentiment_label(row):
    # suma negatywnych emocji
    neg_score = row.get('A_Mean', 0) + row.get('D_Mean', 0) + row.get('S_Mean', 0) + row.get('F_Mean', 0)
    # suma pozytywnych emocji
    pos_score = row.get('H_Mean', 0)
    if pos_score >  neg_score:
        return 1
    else:
        return 0
#nowa kolumna do tabeli entries z zastosowaniem tej funkcji etykietującej
entries['sentiment_label'] = entries.apply(get_sentiment_label, axis=1)

In [ ]:
sns.set_style("whitegrid")

sentiment_colors = {
    'negative': '#E74C3C',
    'neutral':  '#95A5A6',
    'positive': '#2ECC71'
}
plt.figure(figsize=(8, 8))

# agregacja: wystarczy proste zliczenie ile razy wystąpiło 1, a ile razy 0
counts = entries_sample['BERT_Label'].value_counts()

# wykres kołowy
plt.pie(counts,
        labels=counts.index,
        autopct='%1.1f%%',
        colors=[sentiment_colors.get(x, '#333') for x in counts.index],
        startangle=140,
        textprops={'fontsize': 14},
        wedgeprops={'edgecolor': 'white'})

plt.title('Ogólny rozkład sentymentu w próbce (n=2048)', fontsize=16)
plt.show()

party_sentiment = pd.crosstab(entries_sample['club'], entries_sample['BERT_Label'], normalize='index') * 100

# sortowanie względem negatywnego sentymentu
if 'negative' in party_sentiment.columns:
    party_sentiment = party_sentiment.sort_values(by='negative', ascending=False)

#struktura sentymentu dla poszczególnych partii
ax = party_sentiment.plot(kind='bar', stacked=True, figsize=(12, 7),
                          color=[sentiment_colors.get(x, '#333') for x in party_sentiment.columns],
                          edgecolor='white')

plt.title('Struktura sentymentu w Klubach Parlamentarnych (BERT)', fontsize=16)
plt.ylabel('Udział procentowy (%)')
plt.xlabel('Klub')
plt.legend(title='Sentyment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 6))

#histogramy pewności modelu
sns.histplot(data=entries_sample, x='BERT_Score', hue='BERT_Label',
             palette=sentiment_colors, multiple='stack', bins=30)

plt.title('Rozkład pewności modelu (Confidence Score)', fontsize=16)
plt.xlabel('Pewność modelu (0.0 - 1.0)')
plt.ylabel('Liczba wypowiedzi')
plt.show()

Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix


comparison_df = entries_sample[entries_sample['BERT_Label'] != 'neutral'].copy()
#sprawdzamy tylko czy sentyment był klasyfikowany pozytywnie, czy negatywnie
#klasyfikacja de facto binarna, więc trzeba zmapować wartości negative/positive na int
label_map = {'negative': 0, 'positive': 1}
comparison_df['bert_numeric'] = comparison_df['BERT_Label'].map(label_map)
comparison_df = comparison_df.dropna(subset=['sentiment_label', 'bert_numeric'])

#liczenie confusion matrix
cm = confusion_matrix(comparison_df['sentiment_label'], comparison_df['bert_numeric'])

#wizualizacja
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BERT Negatywny', 'BERT Pozytywny'],
            yticklabels=['NAWL Negatywny', 'NAWL Pozytywny'])

plt.ylabel('Corpus based SA')
plt.xlabel('Supervised SA')
plt.title('Confusion Matrix', fontsize=14)
plt.show()

Potencjał dalszego rozwoju:
*   dopisanie gęstości emocjonalnej (tj. klasyfikacja ile słów w wypowiedzi ma zabarwienie emocjonalne w stosunku do wszystkich słów w danej wypowiedzi)
*   odfiltrowanie słów neutralnych, związanych z procesem legislacyjnym

